# Задание 2. Эмпирический закон Ципфа

Проверяется утверждение: длина служебного слова обратно пропорциональна его частоте.

Для каждого словаря берётся список служебных слов (предлоги, союзы, частицы, местоимения, вспомогательные глаголы), который фильтруется по наличию в словаре. Для каждого слова определяются длина $L$ и абсолютная частота $f$. Связь оценивается коэффициентом корреляции Пирсона: отрицательное значение означает, что с увеличением длины слова его частота в среднем уменьшается.


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display, Markdown

ROOT = Path.cwd()
if not (ROOT / 'freq_ru.json').exists():
    ROOT = ROOT.parent
BASE = ROOT / 'rgr2'
DATA = BASE / 'data'
FIG = BASE / 'figures'
DATA.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'font.family': 'DejaVu Sans',
    'font.size': 10,
    'axes.titlesize': 11,
    'axes.labelsize': 10,
    'legend.fontsize': 8,
    'axes.grid': True,
    'grid.alpha': 0.25,
})

FUNCTION_WORDS_RU = [
    "в", "во", "на", "с", "со", "по", "к", "ко", "у", "за", "из", "от", "до", "для", "без", "под", "над", "при", "про",
    "через", "между", "перед", "около", "после", "кроме", "ради", "благодаря",
    "и", "а", "но", "да", "или", "либо", "если", "что", "чтобы", "когда", "пока", "потому", "так", "как", "тоже", "также",
    "ни", "то", "не", "же", "ли", "бы", "ведь", "вот", "вон", "даже", "уже", "еще", "только", "именно", "почти",
    "просто", "разве", "неужели",
    "я", "ты", "он", "она", "оно", "мы", "вы", "они", "меня", "тебя", "его", "ее", "её", "нас", "вас", "их", "мне", "тебе",
    "ему", "ей", "нам", "вам", "им", "себя", "себе", "собой", "себя",
    "этот", "эта", "это", "эти", "тот", "та", "то", "те", "такой", "такая", "такое", "такие", "весь", "вся", "все",
    "свой", "мой", "моя", "моё", "мое", "мои", "твой", "твоё", "твое", "твои", "наш", "наша", "наше", "наши", "ваш",
    "ваша", "ваше", "ваши",
    "кто", "что", "какой", "какая", "какое", "какие", "который", "которая", "которое", "которые", "чей", "чья", "чьё", "чье", "чьи",
    "где", "куда", "откуда", "когда", "почему", "зачем", "сколько",
    "быть", "был", "была", "было", "были", "будет", "буду", "будешь", "будем", "будете", "будут", "есть", "бытие"
]

FUNCTION_WORDS_EN = [
    "the", "a", "an",
    "of", "in", "to", "for", "with", "on", "at", "by", "from", "up", "about", "into", "over", "after", "beneath", "under",
    "above", "across", "along", "among", "around", "before", "behind", "below", "beside", "between", "beyond", "during",
    "except", "inside", "outside", "since", "through", "toward", "towards", "upon", "within", "without",
    "and", "or", "but", "nor", "so", "yet", "because", "although", "though", "if", "unless", "until", "while", "whereas",
    "whether", "as", "than", "that", "this", "these", "those",
    "i", "you", "he", "she", "it", "we", "they", "me", "him", "her", "us", "them", "my", "your", "his", "its", "our", "their",
    "mine", "yours", "hers", "ours", "theirs",
    "myself", "yourself", "himself", "herself", "itself", "ourselves", "yourselves", "themselves",
    "who", "whom", "whose", "which", "what", "where", "when", "why", "how",
    "be", "am", "is", "are", "was", "were", "been", "being",
    "have", "has", "had", "having",
    "do", "does", "did", "doing", "done",
    "will", "would", "shall", "should", "can", "could", "may", "might", "must", "need", "dare", "ought",
    "not", "no", "yes", "too", "very", "just", "only", "also", "even", "well", "there", "here", "now", "then", "thus",
    "hence", "however", "therefore", "already", "still", "again", "ever", "never", "always", "often", "sometimes",
    "once", "soon", "almost", "quite", "rather", "perhaps", "maybe"
]

def dedup_preserve(seq):
    seen = set()
    out = []
    for x in seq:
        if x in seen:
            continue
        seen.add(x)
        out.append(x)
    return out

FUNCTION_WORDS_RU = dedup_preserve(FUNCTION_WORDS_RU)
FUNCTION_WORDS_EN = dedup_preserve(FUNCTION_WORDS_EN)

DICTIONARIES = [
    ('ru_raw',   'Русский, без лемматизации',  'freq_ru.json',   FUNCTION_WORDS_RU),
    ('ru_lemma', 'Русский, с лемматизацией',   'freq_ru_lemma.json', FUNCTION_WORDS_RU),
    ('en_raw',   'Английский, без лемматизации', 'freq_en.json',   FUNCTION_WORDS_EN),
    ('en_lemma', 'Английский, с лемматизацией',  'freq_en_lemma.json', FUNCTION_WORDS_EN),
]

def load_dictionary(filename):
    with open(ROOT / filename, encoding='utf-8') as f:
        package = json.load(f)
    return package, package['dictionary']

packages = {}
for key, label, filename, fw in DICTIONARIES:
    package, dct = load_dictionary(filename)
    packages[key] = {'package': package, 'dict': dct, 'label': label, 'fw': fw}
    print(f"{label}: токенов {package['total_tokens']:,}")


Русский, без лемматизации: токенов 4,982,552
Русский, с лемматизацией: токенов 5,000,251


Английский, без лемматизации: токенов 5,002,249
Английский, с лемматизацией: токенов 5,002,249


In [2]:
rows = []
fw_dfs = {}

for key, label, filename, fw in DICTIONARIES:
    package = packages[key]['package']
    dct = packages[key]['dict']
    total = package['total_tokens']
    data = []
    for w in fw:
        if w in dct:
            f = dct[w]
            data.append({
                'word': w,
                'freq': int(f),
                'length': len(w),
                'rel_freq_per_1e6': f / total * 1e6,
            })
    df = pd.DataFrame(data)
    df['rank_in_dict'] = df['freq'].rank(ascending=False, method='min').astype(int)
    fw_dfs[key] = df
    df.to_csv(DATA / f'function_words_{key}.csv', index=False, float_format='%.6f')

    if len(df) >= 3:
        r, _ = stats.pearsonr(df['length'], df['freq'])
    else:
        r = np.nan

    rows.append({
        'key': key,
        'dictionary': label,
        'n_function_words': int(len(df)),
        'pearson_r': float(r),
    })

task2_summary = pd.DataFrame(rows)
task2_summary.to_csv(DATA / 'zipf2_function_words_summary.csv', index=False, float_format='%.8f')
display(task2_summary.rename(columns={
    'dictionary': 'Словарь',
    'n_function_words': 'Служебных слов',
    'pearson_r': 'r (Пирсон)',
}).round(3))


,key,Словарь,Служебных слов,r (Пирсон)
0,ru_raw,"Русский, без лемматизации",151,-0.311
1,ru_lemma,"Русский, с лемматизацией",93,-0.330
2,en_raw,"Английский, без лемматизации",164,-0.287
3,en_lemma,"Английский, с лемматизацией",145,-0.331


## Графики для задания 2

Показана зависимость относительной частоты служебного слова от его длины. Точки — отдельные слова.


In [3]:
def make_task2_figure(keys, title, out_path):
    fig, axes = plt.subplots(1, 2, figsize=(11.2, 4.4))
    for ax, key in zip(axes, keys):
        df = fw_dfs[key]
        ax.scatter(df['length'], df['rel_freq_per_1e6'], s=28, alpha=0.7,
                   color='#1f77b4')
        ax.set_yscale('log')
        ax.set_title(packages[key]['label'])
        ax.set_xlabel('Длина слова L, символов')
        ax.set_ylabel('Относительная частота, на 10^6 токенов')
        ax.grid(True, alpha=0.25)

    fig.suptitle(title, fontsize=13)
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    fig.savefig(out_path, bbox_inches='tight')
    plt.close(fig)
    print('Сохранено:', out_path)


make_task2_figure(['ru_raw', 'ru_lemma'],
                  'Задание 2. Служебные слова: частота и длина (русский язык)',
                  FIG / 'fig_zipf2_ru.png')
make_task2_figure(['en_raw', 'en_lemma'],
                  'Задание 2. Служебные слова: частота и длина (английский язык)',
                  FIG / 'fig_zipf2_en.png')


Сохранено: /home/dd/nlp_uni/rgr2/figures/fig_zipf2_ru.png


Сохранено: /home/dd/nlp_uni/rgr2/figures/fig_zipf2_en.png


In [4]:
lines = ['### Вывод по заданию 2', '']
for _, row in task2_summary.iterrows():
    lines.append(
        f"- **{row['dictionary']}** (n = {int(row['n_function_words'])}): "
        f"r = {row['pearson_r']:.3f}."
    )

lines += [
    '',
    'Во всех словарях коэффициент корреляции отрицательный: более короткие служебные слова '
    'в среднем встречаются чаще длинных. Это согласуется с эмпирическим законом Ципфа. '
    'Однако связь умеренная, поэтому закон выполняется качественно, а не как строгая '
    'обратная пропорциональность.'
]
display(Markdown('\n'.join(lines)))


### Вывод по заданию 2

- **Русский, без лемматизации** (n = 151): r = -0.311.
- **Русский, с лемматизацией** (n = 93): r = -0.330.
- **Английский, без лемматизации** (n = 164): r = -0.287.
- **Английский, с лемматизацией** (n = 145): r = -0.331.

Во всех словарях коэффициент корреляции отрицательный: более короткие служебные слова в среднем встречаются чаще длинных. Это согласуется с эмпирическим законом Ципфа. Однако связь умеренная, поэтому закон выполняется качественно, а не как строгая обратная пропорциональность.